# Credit Card Fraud Detection — Step 5: Unsupervised Anomaly Detection (Isolation Forest) from Scratch
### Milestone 5 of the 30-Day Capstone Upgrade
**Course:** Python for Data Science (PDS)
**Focus:** First-Principles Unsupervised Outlier Mining — Random Recursive Spatial Partitioning, Average BST Path Length Formulation, and Normalized Anomaly Scoring in Pure NumPy.
**Constraint:** Strictly zero `sklearn` or external libraries — 100% pure NumPy, Pandas, and Matplotlib.

---

## 1. Theoretical Foundations & Mathematical Formulations

### 1.1 Supervised vs. Unsupervised Learning in Financial Fraud
All preceding models (Logistic Regression, Neural Networks, GBDT) operate under the **supervised learning paradigm**, relying on confirmed historical fraud labels. However, modern financial institutions face a critical limitation: **zero-day fraud attacks**.

Fraud rings continuously devise novel transaction obfuscation strategies that have zero prior labels. **Unsupervised Anomaly Detection** models the normal data distribution without labels and flags statistical anomalies purely based on spatial isolation.

### 1.2 Mathematical Formulation of Isolation Forest (Liu, Ting, Zhou, 2008)
Isolation Forest is built on two quantitative realities of anomalous transactions:
1. **Minority:** Anomalies are few in volume ($<0.2\%$).
2. **Attribute-Divergent:** Anomalies possess attribute values starkly different from normal behavioral clusters.

#### The Isolation Tree (iTree)
Given a sub-sample of transactions $\mathcal{X} \subset \mathbb{R}^d$ of size $\psi$ (typically $\psi = 256$):
1. Randomly select an attribute dimension $q \in \{1, \dots, d\}$.
2. Randomly select a cut-point $p \sim \mathcal{U}(\min(\mathcal{X}_q), \max(\mathcal{X}_q))$.
3. Partition the dataset into left and right subsets:
   $$\mathcal{X}_{\text{left}} = \{ \mathbf{x} \in \mathcal{X} : x_q < p \}, \quad \mathcal{X}_{\text{right}} = \{ \mathbf{x} \in \mathcal{X} : x_q \ge p \}$$
4. Recursively build child nodes until the tree reaches height limit $h_{\max} = \lceil \log_2(\psi) \rceil$ or $|\mathcal{X}| \le 1$.

Because anomalous points reside in sparse low-density regions, **they require far fewer random splits to isolate**, resulting in significantly shorter root-to-leaf path lengths $h(\mathbf{x})$.

#### Average Path Length of Unsuccessful Binary Search Tree (BST)
To normalize path length across variable sample sizes $\psi$, Liu et al. utilize the average depth of an unsuccessful search in a Binary Search Tree:
$$c(n) = 2 \left( \ln(n - 1) + \gamma \right) - \frac{2(n - 1)}{n}$$
where $\gamma = 0.5772156649$ is the **Euler-Mascheroni constant**.

#### Anomaly Score Formulation
The anomaly score $s(\mathbf{x}, n)$ of transaction $\mathbf{x}$ across an ensemble of $T$ isolation trees is:
$$s(\mathbf{x}, n) = 2^{-\frac{\mathbb{E}(h(\mathbf{x}))}{c(\psi)}}$$
where $\mathbb{E}(h(\mathbf{x})) = \frac{1}{T} \sum_{t=1}^T h_t(\mathbf{x})$ is the average observed isolation path length.

**Score Interpretations:**
- If $\mathbb{E}(h(\mathbf{x})) \to 0 \implies s(\mathbf{x}, n) \to 1$: **High-confidence anomaly (FRAUD)**.
- If $\mathbb{E}(h(\mathbf{x})) \to c(\psi) \implies s(\mathbf{x}, n) \to 0.5$: Normal instances.
- If $\mathbb{E}(h(\mathbf{x})) \to \psi - 1 \implies s(\mathbf{x}, n) \to 0$: Dense, tightly-packed legitimate transactions.
---

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
print("Environment initialized for Step 5: Isolation Forest from Scratch.")

## 2. Ingesting Data & Preparing Unsupervised Feature Matrix
Notice: Anomaly detection **does not use labels during training**. Labels are strictly held out to evaluate detection accuracy.

In [ ]:
possible_paths = [
    os.path.join('..', 'data', 'creditcard.csv'),
    os.path.join('data', 'creditcard.csv'),
    r'C:\Users\HP\Desktop\PDS\data\creditcard.csv',
    r'C:\Users\HP\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\5518A31AAB1A89BF99B5535DB1EFC4DABE33AAC0\transfers\2026-42\archive\creditcard.csv'
]

DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), None)

if DATA_PATH and os.path.exists(DATA_PATH):
    print(f"Loading official Kaggle dataset from: {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    print("Dataset not found; generating fallback sample dataset.")
    n_samples = 15000
    n_fraud = int(0.002 * n_samples)
    n_legit = n_samples - n_fraud
    time_feat = np.sort(np.random.uniform(0, 172800, n_samples))
    legit_features = np.random.normal(loc=0.0, scale=1.0, size=(n_legit, 28))
    fraud_features = np.random.normal(loc=1.5, scale=1.8, size=(n_fraud, 28))
    features = np.vstack([legit_features, fraud_features])
    legit_amounts = np.random.exponential(scale=85.0, size=n_legit)
    fraud_amounts = np.random.exponential(scale=140.0, size=n_fraud)
    amounts = np.concatenate([legit_amounts, fraud_amounts])
    labels = np.array([0] * n_legit + [1] * n_fraud)
    col_names = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
    data_mat = np.column_stack([time_feat, features, amounts, labels])
    shuffle_idx = np.random.permutation(n_samples)
    df = pd.DataFrame(data_mat[shuffle_idx], columns=col_names)
    df['Class'] = df['Class'].astype(int)

# Cyclic Time and Log Amount Features
hours = (df['Time'] / 3600.0) % 24.0
df['Time_sin'] = np.sin(2.0 * np.pi * hours / 24.0)
df['Time_cos'] = np.cos(2.0 * np.pi * hours / 24.0)
df['Amount_log'] = np.log1p(df['Amount'])

feature_cols = [f'V{i}' for i in range(1, 29)] + ['Time_sin', 'Time_cos', 'Amount_log']
X_all = df[feature_cols].to_numpy(dtype=np.float64)
y_all = df['Class'].to_numpy(dtype=np.int64)

print(f"Total transactions: {len(X_all):,}")
print(f"Ground-truth fraud events: {np.sum(y_all == 1):,} (Outlier ratio: {np.mean(y_all)*100:.3f}%)")

### 2.1 Unsupervised Train/Test Partitioning
We train the Isolation Forest exclusively on **normal, legitimate transactions** (or unlabelled mixed transactions) to model baseline spending patterns, and test its ability to flag held-out fraud attempts.

In [ ]:
np.random.seed(SEED)
idx_0 = np.where(y_all == 0)[0]
idx_1 = np.where(y_all == 1)[0]

# Hold out 30% of fraud and legitimate data for evaluation
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)
n_test_0 = int(len(idx_0) * 0.3)
n_test_1 = int(len(idx_1) * 0.3)

test_idx = np.concatenate([idx_0[:n_test_0], idx_1[:n_test_1]])
train_idx = idx_0[n_test_0:] # Train purely unlabelled on normal operations

X_train_unsupervised = X_all[train_idx]
X_test = X_all[test_idx]
y_test = y_all[test_idx]

print(f"Unsupervised Training Pool: {len(X_train_unsupervised):,} transactions")
print(f"Held-Out Evaluation Split : {len(X_test):,} transactions (Frauds: {np.sum(y_test==1):,}, Legit: {np.sum(y_test==0):,})")

## 3. First-Principles Isolation Forest Implementation in Pure NumPy
Includes:
- `IsolationTreeNode`: Node structure holding split feature, split threshold, size, and depth.
- `IsolationTreeScratch`: Recursive binary partitioning tree with random uniform cuts.
- `IsolationForestScratch`: Forest ensemble with average BST path length calibration.

In [ ]:
EULER_MASCHERONI = 0.5772156649

def c_factor(n):
    """Average path length of unsuccessful search in a BST of n nodes."""
    if n <= 1:
        return 0.0
    if n == 2:
        return 1.0
    return 2.0 * (np.log(n - 1.0) + EULER_MASCHERONI) - (2.0 * (n - 1.0) / n)

class IsolationTreeNode:
    def __init__(self, size, depth):
        self.size = size
        self.depth = depth
        self.split_feature = None
        self.split_val = None
        self.left = None
        self.right = None
        self.is_leaf = False

class IsolationTreeScratch:
    def __init__(self, max_depth):
        self.max_depth = max_depth
        self.root = None
        
    def fit(self, X):
        self.root = self._build(X, depth=0)
        return self
        
    def _build(self, X, depth):
        n_samples, n_features = X.shape
        node = IsolationTreeNode(size=n_samples, depth=depth)
        
        if depth >= self.max_depth or n_samples <= 1:
            node.is_leaf = True
            return node
            
        # Randomly select a feature dimension
        feat_idx = np.random.randint(0, n_features)
        col = X[:, feat_idx]
        min_val, max_val = np.min(col), np.max(col)
        
        if min_val == max_val:
            node.is_leaf = True
            return node
            
        # Random uniform split point between min and max
        split_point = np.random.uniform(min_val, max_val)
        node.split_feature = feat_idx
        node.split_val = split_point
        
        left_mask = col < split_point
        node.left = self._build(X[left_mask], depth + 1)
        node.right = self._build(X[~left_mask], depth + 1)
        return node
        
    def _path_length(self, x, node):
        if node.is_leaf:
            # Unsuccessful search adjustment for remaining node size
            return node.depth + c_factor(node.size)
        if x[node.split_feature] < node.split_val:
            return self._path_length(x, node.left)
        else:
            return self._path_length(x, node.right)
            
    def compute_paths(self, X):
        return np.array([self._path_length(row, self.root) for row in X])

class IsolationForestScratch:
    """
    Complete Isolation Forest (iForest) built from first principles in pure NumPy.
    """
    def __init__(self, n_estimators=100, max_samples=256, seed=42):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.seed = seed
        self.trees = []
        self.c_psi = 0.0
        
    def fit(self, X, verbose=True):
        np.random.seed(self.seed)
        n_total = len(X)
        psi = min(self.max_samples, n_total)
        self.c_psi = c_factor(psi)
        max_depth = int(np.ceil(np.log2(max(psi, 2))))
        
        self.trees = []
        t0 = time.time()
        
        for i in range(self.n_estimators):
            # Subsample without replacement
            sample_indices = np.random.choice(n_total, size=psi, replace=False)
            X_sub = X[sample_indices]
            
            tree = IsolationTreeScratch(max_depth=max_depth)
            tree.fit(X_sub)
            self.trees.append(tree)
            
            if verbose and ((i + 1) % 25 == 0 or i == 0):
                print(f"   Constructed {i+1:3d}/{self.n_estimators} isolation trees...")
                
        if verbose:
            print(f"Forest built with {self.n_estimators} trees in {time.time()-t0:.2f}s (h_max = {max_depth})")
        return self
        
    def anomaly_score(self, X):
        """
        Computes s(x, n) = 2^(-E(h(x)) / c(psi)).
        Scores close to 1 indicate strong anomalies.
        """
        all_paths = np.zeros(len(X), dtype=np.float64)
        for tree in self.trees:
            all_paths += tree.compute_paths(X)
        mean_paths = all_paths / float(len(self.trees))
        scores = 2.0 ** (-mean_paths / self.c_psi)
        return scores
        
    def predict(self, X, threshold=0.60):
        """Classifies samples with score >= threshold as Anomalies (1), else Normal (0)."""
        return (self.anomaly_score(X) >= threshold).astype(int)

print("IsolationForestScratch defined successfully in pure NumPy.")

## 4. Unsupervised Model Training
We construct an ensemble of $T = 100$ isolation trees using subsample size $\psi = 256$.

In [ ]:
iforest = IsolationForestScratch(n_estimators=100, max_samples=256, seed=SEED)
print("Fitting Isolation Forest on unlabelled normal transaction pool...")
iforest.fit(X_train_unsupervised, verbose=True)

## 5. Empirical Evaluation on Held-Out Transactions
We test if the unsupervised anomaly score effectively discriminates fraudulent outliers without ever seeing a label during training.

In [ ]:
t_eval0 = time.time()
print("Computing anomaly scores on held-out test split...")
anomaly_scores = iforest.anomaly_score(X_test)
print(f"Scores computed in {time.time() - t_eval0:.2f}s!")

# Separate scores by true label
scores_fraud = anomaly_scores[y_test == 1]
scores_legit = anomaly_scores[y_test == 0]

print(f"\nMean Anomaly Score for Legitimate Transactions : {np.mean(scores_legit):.4f} ± {np.std(scores_legit):.4f}")
print(f"Mean Anomaly Score for Fraudulent Transactions : {np.mean(scores_fraud):.4f} ± {np.std(scores_fraud):.4f}")

# First-principles trapezoidal quadrature
def trapezoid_scratch(y, x):
    return float(np.sum(0.5 * (y[:-1] + y[1:]) * np.diff(x)))

# Sweep thresholds across anomaly score spectrum
th_grid = np.linspace(np.min(anomaly_scores), np.max(anomaly_scores), 201)
tprs, fprs = [], []
P = np.sum(y_test == 1)
N = np.sum(y_test == 0)

for th in th_grid:
    y_pred = (anomaly_scores >= th).astype(int)
    tprs.append(np.sum((y_test == 1) & (y_pred == 1)) / P)
    fprs.append(np.sum((y_test == 0) & (y_pred == 1)) / N)

fpr_arr = np.array(fprs)
tpr_arr = np.array(tprs)
sort_idx = np.argsort(fpr_arr)
auc_unsupervised = trapezoid_scratch(tpr_arr[sort_idx], fpr_arr[sort_idx])

print(f"\nUnsupervised Anomaly Detection ROC-AUC: {auc_unsupervised:.4f}")

### 5.1 Visualizing Anomaly Score Separation & Unsupervised ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Anomaly Score Distribution Comparison
axes[0].hist(scores_legit, bins=50, alpha=0.6, color='#1f77b4', density=True, label='Normal Legitimate')
axes[0].hist(scores_fraud, bins=50, alpha=0.7, color='#d62728', density=True, label='Fraudulent Outliers')
axes[0].axvline(0.50, color='black', linestyle='--', linewidth=1.5, label='Normal Baseline (s=0.5)')
axes[0].set_xlabel('Unsupervised Anomaly Score s(x, n)')
axes[0].set_ylabel('Probability Density')
axes[0].set_title('Clear Spatial Separation of Fraud Outliers (Zero Labels Used)')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# 2. Unsupervised ROC Curve
axes[1].plot(fpr_arr[sort_idx], tpr_arr[sort_idx], color='#9467bd', linewidth=2.5, label=f'Isolation Forest (AUC = {auc_unsupervised:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random Chance (AUC = 0.50)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall / Sensitivity)')
axes[1].set_title('Unsupervised Outlier Detection ROC Frontier')
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Academic & Methodological Takeaways

1. **Unsupervised Defense Against Zero-Day Attacks:**
   The from-scratch Isolation Forest demonstrates that even without access to fraudulent training labels ($y = 1$), anomalous transactions are isolated significantly faster (shorter path lengths) than dense legitimate purchase clusters, producing a strong separation in anomaly score $s(\mathbf{x}, n)$.
2. **Theoretical Precision:**
   The calibration factor $c(n) = 2(\ln(n-1) + \gamma) - \frac{2(n-1)}{n}$ based on Binary Search Tree properties correctly normalizes path lengths across trees regardless of sub-sample sizes.
3. **Zero Package Dependencies:**
   Random recursive spatial partition splitting, BST path length accumulation, and Euler-Mascheroni normalized scoring were implemented 100% in pure NumPy.